# Phase-by-Phase Model Probe

Each scenario cell below builds one controlled traffic situation, feeds the corresponding 10-value state vector to the trained DQN, and displays both the lane-level queue table and the model output.

The model only receives aggregate approach queues and wait times, so the left/straight/right lane counts are shown for human interpretation and then summed into `A/B/C/D` queues before inference.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

source_root = str(project_root / "code")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from State.traffic_env import TrafficEnv
from Neural_Networks.DQN_Implementation.dqn import DQN

env = TrafficEnv()
PHASE_NAMES = env.phases
APPROACHES = ["A", "B", "C", "D"]
TURNS = ["left", "straight", "right"]

model = DQN(len(env._get_state()), len(PHASE_NAMES))
model_path = project_root / "code" / "Neural_Networks" / "DQN_Implementation" / "traffic_dqn_model.pth"
model.load_state_dict(torch.load(model_path, map_location=torch.device("cpu")))
model.eval()


DQN(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=4, bias=True)
  )
)

In [2]:
def make_state(lanes, waits, current_phase=0, time_in_phase=0):
    queues = np.array([sum(lanes[a].values()) for a in APPROACHES], dtype=float)
    waits = np.array([waits.get(a, 0) for a in APPROACHES], dtype=float)
    return np.concatenate([
        queues / 20.0,
        waits / 50.0,
        [current_phase / len(PHASE_NAMES)],
        [time_in_phase / 50.0],
    ])

def predict(state):
    with torch.no_grad():
        q_values = model(torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)).squeeze(0).numpy()
    action = int(np.argmax(q_values))
    return action, q_values

def scenario_table(lanes, waits, current_phase, time_in_phase):
    rows = []
    for approach in APPROACHES:
        lane_counts = lanes[approach]
        rows.append({
            "approach": approach,
            "left_queue": lane_counts.get("left", 0),
            "straight_queue": lane_counts.get("straight", 0),
            "right_queue": lane_counts.get("right", 0),
            "total_queue": sum(lane_counts.values()),
            "wait_time_s": waits.get(approach, 0),
            "current_phase": PHASE_NAMES[current_phase],
            "time_in_phase_s": time_in_phase,
        })
    return pd.DataFrame(rows)

def q_table(q_values, action):
    return pd.DataFrame({
        "phase_id": list(range(len(PHASE_NAMES))),
        "phase": PHASE_NAMES,
        "q_value": q_values,
        "selected": [i == action for i in range(len(PHASE_NAMES))],
    })

def run_scenario(name, lanes, waits=None, current_phase=0, time_in_phase=0):
    waits = waits or {a: 0 for a in APPROACHES}
    state = make_state(lanes, waits, current_phase, time_in_phase)
    action, q_values = predict(state)
    display(Markdown(f"## {name}"))
    display(Markdown(f"**Selected action:** `{action}` / `{PHASE_NAMES[action]}`"))
    display(Markdown("**Lane table shown to us, then aggregated into the model input:**"))
    display(scenario_table(lanes, waits, current_phase, time_in_phase))
    display(Markdown("**Normalized model input:**"))
    display(pd.DataFrame([state], columns=["A_queue", "B_queue", "C_queue", "D_queue", "A_wait", "B_wait", "C_wait", "D_wait", "current_phase", "time_in_phase"]))
    display(Markdown("**Model output Q-values:**"))
    display(q_table(q_values, action))
    return action, q_values


In [3]:
# Situation 1: balanced light traffic on all approaches.
balanced_lanes = {
    "A": {"left": 1, "straight": 2, "right": 1},
    "B": {"left": 1, "straight": 2, "right": 1},
    "C": {"left": 1, "straight": 2, "right": 1},
    "D": {"left": 1, "straight": 2, "right": 1},
}
balanced_waits = {"A": 6, "B": 6, "C": 6, "D": 6}
run_scenario("Situation 1: balanced queues", balanced_lanes, balanced_waits, current_phase=0, time_in_phase=8)


## Situation 1: balanced queues

**Selected action:** `0` / `AC_forward`

**Lane table shown to us, then aggregated into the model input:**

,approach,left_queue,straight_queue,right_queue,total_queue,wait_time_s,current_phase,time_in_phase_s
0,A,1,2,1,4,6,AC_forward,8
1,B,1,2,1,4,6,AC_forward,8
2,C,1,2,1,4,6,AC_forward,8
3,D,1,2,1,4,6,AC_forward,8


**Normalized model input:**

,A_queue,B_queue,C_queue,D_queue,A_wait,B_wait,C_wait,D_wait,current_phase,time_in_phase
0,0.2,0.2,0.2,0.2,0.12,0.12,0.12,0.12,0.0,0.16


**Model output Q-values:**

,phase_id,phase,q_value,selected
0,0,AC_forward,-990.126770,True
1,1,BD_forward,-991.445251,False
2,2,AC_left,-1028.942993,False
3,3,BD_left,-1024.518555,False


(0, array([ -990.1268 ,  -991.44525, -1028.943  , -1024.5186 ], dtype=float32))

In [4]:
# Situation 2: A/C through traffic is heavy, so AC_forward should be attractive if the model learned demand service.
ac_through_lanes = {
    "A": {"left": 1, "straight": 9, "right": 3},
    "B": {"left": 0, "straight": 2, "right": 1},
    "C": {"left": 2, "straight": 8, "right": 2},
    "D": {"left": 1, "straight": 1, "right": 1},
}
ac_through_waits = {"A": 18, "B": 4, "C": 16, "D": 5}
run_scenario("Situation 2: heavy A/C straight and right demand", ac_through_lanes, ac_through_waits, current_phase=1, time_in_phase=20)


## Situation 2: heavy A/C straight and right demand

**Selected action:** `0` / `AC_forward`

**Lane table shown to us, then aggregated into the model input:**

,approach,left_queue,straight_queue,right_queue,total_queue,wait_time_s,current_phase,time_in_phase_s
0,A,1,9,3,13,18,BD_forward,20
1,B,0,2,1,3,4,BD_forward,20
2,C,2,8,2,12,16,BD_forward,20
3,D,1,1,1,3,5,BD_forward,20


**Normalized model input:**

,A_queue,B_queue,C_queue,D_queue,A_wait,B_wait,C_wait,D_wait,current_phase,time_in_phase
0,0.65,0.15,0.6,0.15,0.36,0.08,0.32,0.1,0.25,0.4


**Model output Q-values:**

,phase_id,phase,q_value,selected
0,0,AC_forward,-1333.572388,True
1,1,BD_forward,-1365.333130,False
2,2,AC_left,-1390.939087,False
3,3,BD_left,-1394.824951,False


(0, array([-1333.5724, -1365.3331, -1390.9391, -1394.825 ], dtype=float32))

In [5]:
# Situation 3: B/D through traffic dominates.
bd_through_lanes = {
    "A": {"left": 1, "straight": 1, "right": 0},
    "B": {"left": 2, "straight": 10, "right": 3},
    "C": {"left": 0, "straight": 2, "right": 1},
    "D": {"left": 2, "straight": 9, "right": 2},
}
bd_through_waits = {"A": 3, "B": 21, "C": 5, "D": 19}
run_scenario("Situation 3: heavy B/D straight and right demand", bd_through_lanes, bd_through_waits, current_phase=0, time_in_phase=18)


## Situation 3: heavy B/D straight and right demand

**Selected action:** `1` / `BD_forward`

**Lane table shown to us, then aggregated into the model input:**

,approach,left_queue,straight_queue,right_queue,total_queue,wait_time_s,current_phase,time_in_phase_s
0,A,1,1,0,2,3,AC_forward,18
1,B,2,10,3,15,21,AC_forward,18
2,C,0,2,1,3,5,AC_forward,18
3,D,2,9,2,13,19,AC_forward,18


**Normalized model input:**

,A_queue,B_queue,C_queue,D_queue,A_wait,B_wait,C_wait,D_wait,current_phase,time_in_phase
0,0.1,0.75,0.15,0.65,0.06,0.42,0.1,0.38,0.0,0.36


**Model output Q-values:**

,phase_id,phase,q_value,selected
0,0,AC_forward,-1439.574829,False
1,1,BD_forward,-1404.309448,True
2,2,AC_left,-1478.384155,False
3,3,BD_left,-1458.715210,False


(1, array([-1439.5748, -1404.3094, -1478.3842, -1458.7152], dtype=float32))

In [6]:
# Situation 4: A/C left-turn pockets are backing up.
ac_left_lanes = {
    "A": {"left": 8, "straight": 2, "right": 1},
    "B": {"left": 1, "straight": 2, "right": 1},
    "C": {"left": 7, "straight": 1, "right": 1},
    "D": {"left": 1, "straight": 2, "right": 0},
}
ac_left_waits = {"A": 24, "B": 7, "C": 23, "D": 8}
run_scenario("Situation 4: protected A/C left-turn pressure", ac_left_lanes, ac_left_waits, current_phase=0, time_in_phase=25)


## Situation 4: protected A/C left-turn pressure

**Selected action:** `0` / `AC_forward`

**Lane table shown to us, then aggregated into the model input:**

,approach,left_queue,straight_queue,right_queue,total_queue,wait_time_s,current_phase,time_in_phase_s
0,A,8,2,1,11,24,AC_forward,25
1,B,1,2,1,4,7,AC_forward,25
2,C,7,1,1,9,23,AC_forward,25
3,D,1,2,0,3,8,AC_forward,25


**Normalized model input:**

,A_queue,B_queue,C_queue,D_queue,A_wait,B_wait,C_wait,D_wait,current_phase,time_in_phase
0,0.55,0.2,0.45,0.15,0.48,0.14,0.46,0.16,0.0,0.5


**Model output Q-values:**

,phase_id,phase,q_value,selected
0,0,AC_forward,-1260.049927,True
1,1,BD_forward,-1289.841431,False
2,2,AC_left,-1314.660034,False
3,3,BD_left,-1318.270508,False


(0, array([-1260.0499, -1289.8414, -1314.66  , -1318.2705], dtype=float32))

In [7]:
# Situation 5: B/D left-turn pockets are backing up.
bd_left_lanes = {
    "A": {"left": 1, "straight": 2, "right": 1},
    "B": {"left": 9, "straight": 1, "right": 1},
    "C": {"left": 1, "straight": 2, "right": 1},
    "D": {"left": 8, "straight": 2, "right": 1},
}
bd_left_waits = {"A": 6, "B": 28, "C": 7, "D": 26}
run_scenario("Situation 5: protected B/D left-turn pressure", bd_left_lanes, bd_left_waits, current_phase=1, time_in_phase=24)


## Situation 5: protected B/D left-turn pressure

**Selected action:** `1` / `BD_forward`

**Lane table shown to us, then aggregated into the model input:**

,approach,left_queue,straight_queue,right_queue,total_queue,wait_time_s,current_phase,time_in_phase_s
0,A,1,2,1,4,6,BD_forward,24
1,B,9,1,1,11,28,BD_forward,24
2,C,1,2,1,4,7,BD_forward,24
3,D,8,2,1,11,26,BD_forward,24


**Normalized model input:**

,A_queue,B_queue,C_queue,D_queue,A_wait,B_wait,C_wait,D_wait,current_phase,time_in_phase
0,0.2,0.55,0.2,0.55,0.12,0.56,0.14,0.52,0.25,0.48


**Model output Q-values:**

,phase_id,phase,q_value,selected
0,0,AC_forward,-1377.950317,False
1,1,BD_forward,-1347.075684,True
2,2,AC_left,-1416.123657,False
3,3,BD_left,-1398.351562,False


(1, array([-1377.9503, -1347.0757, -1416.1237, -1398.3516], dtype=float32))

In [8]:
# Situation 6: starvation check. A modest B/D queue has waited much longer than larger A/C traffic.
starvation_lanes = {
    "A": {"left": 2, "straight": 6, "right": 2},
    "B": {"left": 1, "straight": 3, "right": 1},
    "C": {"left": 2, "straight": 6, "right": 2},
    "D": {"left": 1, "straight": 3, "right": 1},
}
starvation_waits = {"A": 9, "B": 42, "C": 10, "D": 40}
run_scenario("Situation 6: B/D starvation pressure", starvation_lanes, starvation_waits, current_phase=0, time_in_phase=30)


## Situation 6: B/D starvation pressure

**Selected action:** `1` / `BD_forward`

**Lane table shown to us, then aggregated into the model input:**

,approach,left_queue,straight_queue,right_queue,total_queue,wait_time_s,current_phase,time_in_phase_s
0,A,2,6,2,10,9,AC_forward,30
1,B,1,3,1,5,42,AC_forward,30
2,C,2,6,2,10,10,AC_forward,30
3,D,1,3,1,5,40,AC_forward,30


**Normalized model input:**

,A_queue,B_queue,C_queue,D_queue,A_wait,B_wait,C_wait,D_wait,current_phase,time_in_phase
0,0.5,0.25,0.5,0.25,0.18,0.84,0.2,0.8,0.0,0.6


**Model output Q-values:**

,phase_id,phase,q_value,selected
0,0,AC_forward,-1394.076538,False
1,1,BD_forward,-1361.643555,True
2,2,AC_left,-1432.761719,False
3,3,BD_left,-1414.395020,False


(1, array([-1394.0765, -1361.6436, -1432.7617, -1414.395 ], dtype=float32))